# BirdCLEF 2026 Training v34 — Pseudo-Label Fine-tuning

## Strategy
Extend training data by adding **pseudo-labeled test soundscapes** from v30 inference.
- Ground-truth: 66 training soundscapes (hard labels)
- Pseudo-labeled: test soundscapes (v30 soft-label predictions, weight=0.5)
Warm-start from v30 checkpoints, train at `lr=1e-5` for `10 epochs`.

## Why this works
The model sees the **test distribution** during training — test soundscapes
represent the exact acoustic conditions it will be evaluated on.

## Required Kaggle inputs
1. `birdclef-2026` (train_soundscapes_labels.csv, taxonomy.csv)
2. `chiragggg/birdclef-2026-perch-embs-v3` (train embeddings)
3. `chiragggg/birdclef-2026-perch-weights-v30` (warm-start checkpoints)
4. `chiragggg/birdclef-2026-test-perch-embs-v34` (test window embeddings)
5. `chiragggg/birdclef-2026-pseudo-labels-v34` (soft labels pickle)

## Output
Upload as `chiragggg/birdclef-2026-perch-weights-v34`
Files: `perch_gru_v34_fold0.pt` ... `perch_gru_v34_fold4.pt`


In [ ]:
# === CELL 1: IMPORTS & CONFIG ===
import os, json, copy, random
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm

CFG = dict(
    folds           = 5,
    epochs          = 10,          # fine-tune with pseudo-labeled data
    warmup_epochs   = 1,
    lr              = 1e-5,        # higher than v30 since we have more data now
    batch_size      = 4,           # sequences per batch (each ~24 windows)
    num_workers     = 2,
    seed            = 42,
    perch_emb_dim   = 1536,
    perch_emb_noise = 0.01,        # less noise than v23 (0.02) during fine-tune
    gru_hidden      = 512,         # MUST match v23 exactly
    gru_layers      = 2,           # MUST match v23 exactly
    gru_dropout     = 0.3,
    max_seq_len     = 24,          # max windows per soundscape (2 min @ 5 s = 24)
    checkpoint_tag  = 'v34',
    device          = 'cuda' if torch.cuda.is_available() else 'cpu',
)

random.seed(CFG['seed'])
np.random.seed(CFG['seed'])
torch.manual_seed(CFG['seed'])
device = torch.device(CFG['device'])

print(f"v34 Pseudo-Label Fine-tuning")
print(f"  Device   : {device}")
print(f"  Epochs   : {CFG['epochs']}  LR={CFG['lr']}  pseudo_weight={CFG['pseudo_weight']}")
print(f"  GRU arch : hidden={CFG['gru_hidden']}, layers={CFG['gru_layers']} (same as v30)")

In [ ]:
# === CELL 2: PATHS & SPECIES ===
def _fe(*candidates):
    return next((p for p in candidates if os.path.exists(p)), candidates[0])

# Competition data
TAXONOMY_CSV    = _fe('/kaggle/input/birdclef-2026/taxonomy.csv',
                      '/kaggle/input/competitions/birdclef-2026/taxonomy.csv')
SOUNDSCAPE_ANNO = _fe('/kaggle/input/birdclef-2026/train_soundscapes_labels.csv',
                      '/kaggle/input/competitions/birdclef-2026/train_soundscapes_labels.csv')

# Pre-computed Perch embeddings
EMBD_DIR = _fe(
    '/kaggle/input/birdclef-2026-perch-embs-v3/perch_embeddings_v3',
    '/kaggle/input/birdclef-2026-perch-embs-v3',
    '/kaggle/input/datasets/chiragggg/birdclef-2026-perch-embs-v3/perch_embeddings_v3',
    '/kaggle/input/datasets/chiragggg/birdclef-2026-perch-embs-v3',
)

# v30 checkpoints to warm-start from
V30_CKPT_DIR = _fe(
    '/kaggle/input/birdclef-2026-perch-weights-v30',
    '/kaggle/input/datasets/chiragggg/birdclef-2026-perch-weights-v30',
)

OUT_DIR = '/kaggle/working'
os.makedirs(OUT_DIR, exist_ok=True)

# Pseudo-label inputs
TEST_EMBD_DIR = _fe(
    '/kaggle/input/birdclef-2026-test-perch-embs-v34',
    '/kaggle/input/datasets/chiragggg/birdclef-2026-test-perch-embs-v34',
)
PSEUDO_LABELS_PATH = _fe(
    '/kaggle/input/birdclef-2026-pseudo-labels-v34/pseudo_labels_v34.pkl',
    '/kaggle/input/datasets/chiragggg/birdclef-2026-pseudo-labels-v34/pseudo_labels_v34.pkl',
)
print(f'TEST_EMBD_DIR     : {TEST_EMBD_DIR}')
print(f'PSEUDO_LABELS     : {PSEUDO_LABELS_PATH}')

taxonomy_df = pd.read_csv(TAXONOMY_CSV)
species     = taxonomy_df['primary_label'].astype(str).tolist()
sp_idx      = {lab: i for i, lab in enumerate(species)}
n_classes   = len(species)

_all_emb = list(Path(EMBD_DIR).glob('soundscape_*.npy')) if os.path.isdir(EMBD_DIR) else []
print(f"Species            : {n_classes}")
print(f"EMBD_DIR           : {EMBD_DIR}")
print(f"  soundscape .npy  : {len(_all_emb)}")
print(f"V30_CKPT_DIR       : {V30_CKPT_DIR}")
_v30_ckpts = list(Path(V30_CKPT_DIR).glob('perch_gru_v30_fold*.pt')) if os.path.isdir(V30_CKPT_DIR) else []
print(f"  v30 checkpoints  : {sorted([p.name for p in _v30_ckpts])}")

In [ ]:
# === CELL 3: LABEL HELPERS ===
def soundscape_to_multihot(label_str):
    '''Convert semicolon-separated taxon-ID string to multi-hot vector.'''
    y = np.zeros(n_classes, dtype='float32')
    for sp in str(label_str).split(';'):
        sp = sp.strip()
        if sp in sp_idx:
            y[sp_idx[sp]] = 1.0
    return y

def _parse_hms(s):
    '''HH:MM:SS -> total seconds.'''
    p = str(s).strip().split(':')
    return int(p[0]) * 3600 + int(p[1]) * 60 + int(p[2])

print('Label helpers defined')

In [ ]:
# === CELL 4: PERCHGRU MODEL (identical architecture to v23) ===
# Architecture must match v23 exactly so checkpoint weights load correctly.
class PerchGRU(nn.Module):
    '''Bidirectional GRU over per-window Perch 1536-d embeddings.
    Input : (B, T, 1536) padded sequence  or  (B, 1536) single window.
    Output: (B, T, n_classes) logits.
    Architecture identical to v23 (hidden=512, layers=2).
    '''
    def __init__(self, n_classes, emb_dim=1536, hidden=512, n_layers=2, dropout=0.3):
        super().__init__()
        self.proj = nn.Sequential(
            nn.LayerNorm(emb_dim),
            nn.Linear(emb_dim, 512),
            nn.GELU(),
        )
        self.gru = nn.GRU(
            512, hidden, n_layers,
            batch_first=True, bidirectional=True,
            dropout=dropout if n_layers > 1 else 0.0,
        )
        self.head = nn.Sequential(
            nn.LayerNorm(hidden * 2),
            nn.Dropout(0.2),
            nn.Linear(hidden * 2, n_classes),
        )

    def forward(self, x):
        single = (x.dim() == 2)
        if single:
            x = x.unsqueeze(1)
        h, _ = self.gru(self.proj(x))
        out   = self.head(h)
        return out.squeeze(1) if single else out


# Quick shape check
_m = PerchGRU(n_classes).to(device)
_x = torch.randn(2, 8, 1536).to(device)
assert _m(_x).shape == (2, 8, n_classes), "Shape mismatch!"
del _m, _x
print(f"PerchGRU OK  (hidden={CFG['gru_hidden']}, layers={CFG['gru_layers']})")

In [ ]:
# === CELL 5: SOUNDSCAPE SEQUENCE DATASET (pseudo-label aware) ===
import pickle

class SoundscapeSeqDataset(Dataset):
    '''Each item is one soundscape sequence.
    Supports two emb_roots: one for train, one for pseudo-labeled test.
    '''
    def __init__(self, seq_groups, emb_root, train=True, pseudo_weight=1.0,
                 extra_emb_root=None):
        self.groups        = seq_groups
        self.emb_root      = Path(emb_root)
        self.extra_emb_root = Path(extra_emb_root) if extra_emb_root else None
        self.train         = train
        self.pseudo_weight = pseudo_weight  # per-item loss weight

    def __len__(self):
        return len(self.groups)

    def _load_emb(self, stem):
        """Try main emb_root first, then extra_emb_root."""
        ep = self.emb_root / (stem + '.npy')
        if ep.exists(): return np.load(str(ep)).astype('float32')
        if self.extra_emb_root is not None:
            ep2 = self.extra_emb_root / (stem + '.npy')
            if ep2.exists(): return np.load(str(ep2)).astype('float32')
        return np.zeros(CFG['perch_emb_dim'], dtype='float32')

    def __getitem__(self, i):
        grp     = self.groups[i]
        windows = sorted(grp['windows'], key=lambda w: w[1])[:CFG['max_seq_len']]
        T       = len(windows)
        embs    = np.zeros((T, CFG['perch_emb_dim']), dtype='float32')
        labels  = np.zeros((T, n_classes), dtype='float32')
        for t, (stem, end_secs, lv) in enumerate(windows):
            e = self._load_emb(stem)
            if self.train and random.random() < 0.5:
                e = e + np.random.randn(*e.shape).astype('float32') * CFG['perch_emb_noise']
            embs[t]   = e
            labels[t] = lv
        x = torch.from_numpy(embs)    # (T, 1536)
        y = torch.from_numpy(labels)  # (T, n_classes)
        w = torch.tensor(self.pseudo_weight, dtype=torch.float32)
        return x, y, w


def seq_collate(batch):
    """Pad variable-length sequences; return (x_pad, y_pad, mask, weights)."""
    xs, ys, ws = zip(*batch)
    max_T  = max(x.shape[0] for x in xs)
    B      = len(xs)
    x_pad  = torch.zeros(B, max_T, CFG['perch_emb_dim'])
    y_pad  = torch.zeros(B, max_T, n_classes)
    mask   = torch.zeros(B, max_T, dtype=torch.bool)
    for i, (x, y) in enumerate(zip(xs, ys)):
        T = x.shape[0]
        x_pad[i, :T] = x
        y_pad[i, :T] = y
        mask[i, :T]  = True
    weights = torch.stack(ws)  # (B,)
    return x_pad, y_pad, mask, weights


print('SoundscapeSeqDataset + seq_collate defined')


In [ ]:
# === CELL 6: BUILD COMBINED SEQUENCE GROUPS ===

# Ground-truth training groups
sc_anno = pd.read_csv(SOUNDSCAPE_ANNO)
_sc_label_map = {}
for _, row in sc_anno.iterrows():
    sc_stem  = Path(str(row['filename'])).stem
    end_secs = _parse_hms(row['end'])
    lv       = soundscape_to_multihot(row['primary_label'])
    _sc_label_map[(sc_stem, end_secs)] = lv

_sc_groups      = defaultdict(list)
_missing_labels = 0
for f in Path(EMBD_DIR).glob('soundscape_*.npy'):
    try: stem_part, end_part = f.stem.rsplit('_', 1)
    except ValueError: continue
    if not end_part.endswith('s'): continue
    end_secs = int(end_part[:-1])
    sc_stem  = stem_part[len('soundscape_'):]
    lv = _sc_label_map.get((sc_stem, end_secs))
    if lv is None: _missing_labels += 1; continue
    _sc_groups[sc_stem].append((f.stem, end_secs, lv))

gt_groups = [
    {'stem': sc_stem, 'windows': windows}
    for sc_stem, windows in _sc_groups.items() if windows
]
print(f'Ground-truth sequences : {len(gt_groups)}')
print(f'  Total windows        : {sum(len(g["windows"]) for g in gt_groups)}')

# Pseudo-labeled test groups
if os.path.exists(PSEUDO_LABELS_PATH):
    with open(PSEUDO_LABELS_PATH, 'rb') as f:
        pseudo_groups = pickle.load(f)
    print(f'Pseudo-label sequences : {len(pseudo_groups)}')
    print(f'  Total windows        : {sum(len(g["windows"]) for g in pseudo_groups)}')
else:
    pseudo_groups = []
    print('WARNING: pseudo_labels_v34.pkl not found -- training on ground-truth only')

print(f'Combined               : {len(gt_groups) + len(pseudo_groups)} sequences')


In [ ]:
# === CELL 7: 5-FOLD FINE-TUNING FROM v30 WEIGHTS (PSEUDO-LABEL) ===
# Each fold:
#   1. Load perch_gru_v30_fold{i}.pt
#   2. Fine-tune on all soundscape sequences for CFG['epochs'] epochs
#   3. Save perch_gru_v34_fold{i}.pt (best loss checkpoint)

print('=' * 65)
print(f"v34 Pseudo-Label Fine-tuning  {CFG['folds']} folds  AMP={torch.cuda.is_available()}")
print(f"LR={CFG['lr']}  Epochs={CFG['epochs']}  Batch={CFG['batch_size']}")
print('=' * 65)

_use_amp   = (device.type == 'cuda')
_criterion = nn.BCEWithLogitsLoss(reduction='none')

# Dataset is the same for all folds (all 66 soundscapes)
# Ground-truth dataset (weight=1.0)
gt_ds = SoundscapeSeqDataset(gt_groups, EMBD_DIR, train=True,
                              pseudo_weight=1.0, extra_emb_root=TEST_EMBD_DIR)
# Pseudo-labeled dataset (weight=pseudo_weight)
ps_ds = SoundscapeSeqDataset(pseudo_groups, TEST_EMBD_DIR, train=True,
                              pseudo_weight=CFG['pseudo_weight'], extra_emb_root=EMBD_DIR)
from torch.utils.data import ConcatDataset
sc_ds = ConcatDataset([gt_ds, ps_ds]) if pseudo_groups else gt_ds
sc_dl = DataLoader(
    sc_ds,
    batch_size=CFG['batch_size'],
    shuffle=True,
    num_workers=CFG['num_workers'],
    collate_fn=seq_collate,
    drop_last=False,
    pin_memory=_use_amp,
)
print(f"DataLoader: {len(sc_ds)} combined sequences  {len(sc_dl)} batches/epoch")

fold_results = []

for fold_idx in range(CFG['folds']):
    v30_ckpt = Path(V30_CKPT_DIR) / f"perch_gru_v30_fold{fold_idx}.pt"
    if not v30_ckpt.exists():
        print(f"[SKIP] v30 checkpoint not found: {v30_ckpt}")
        continue

    print(f"\nFold {fold_idx + 1}/{CFG['folds']}  loading {v30_ckpt.name}")

    model = PerchGRU(
        n_classes, CFG['perch_emb_dim'],
        CFG['gru_hidden'], CFG['gru_layers'], CFG['gru_dropout'],
    ).to(device)
    model.load_state_dict(torch.load(v30_ckpt, map_location=device, weights_only=True))
    print("  Loaded v30 weights")

    optimizer = AdamW(model.parameters(), lr=CFG['lr'], weight_decay=1e-4)
    scaler    = GradScaler(enabled=_use_amp)

    # Short warmup then cosine decay
    warmup_sched = LinearLR(optimizer, start_factor=0.3, end_factor=1.0,
                            total_iters=CFG['warmup_epochs'])
    cosine_sched = CosineAnnealingLR(
        optimizer,
        T_max=max(1, CFG['epochs'] - CFG['warmup_epochs']),
        eta_min=1e-7,
    )
    scheduler = SequentialLR(optimizer,
                             schedulers=[warmup_sched, cosine_sched],
                             milestones=[CFG['warmup_epochs']])

    best_loss  = float('inf')
    best_state = None

    for epoch in range(CFG['epochs']):
        model.train()
        ep_loss   = 0.0
        n_batches = 0

        for x_pad, y_pad, mask, sample_w in tqdm(sc_dl, desc=f"  Ep {epoch + 1}", leave=False):
            x_pad    = x_pad.to(device)
            y_pad    = y_pad.to(device)
            mask     = mask.to(device)
            sample_w = sample_w.to(device)  # (B,)

            optimizer.zero_grad()
            with autocast(enabled=_use_amp):
                logits = model(x_pad)                          # (B, T, C)
                loss_e = _criterion(logits, y_pad)             # (B, T, C)
                m      = mask.unsqueeze(-1).float()            # (B, T, 1)
                sw     = sample_w.view(-1, 1, 1)  # (B, 1, 1)
                loss   = (loss_e * m * sw).sum() / (m * sw).sum().clamp(min=1)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()

            ep_loss   += loss.item()
            n_batches += 1

        ep_loss /= max(n_batches, 1)
        scheduler.step()

        if ep_loss < best_loss:
            best_loss  = ep_loss
            best_state = copy.deepcopy(model.state_dict())

        print(f"  Ep {epoch + 1:2d}/{CFG['epochs']}  loss={ep_loss:.4f}")

    # Restore best and save
    if best_state is not None:
        model.load_state_dict(best_state)
    out_ckpt = os.path.join(OUT_DIR, f"perch_gru_v34_fold{fold_idx}.pt")
    torch.save(model.state_dict(), out_ckpt)
    fold_results.append(best_loss)
    print(f"  Saved {out_ckpt}  best_loss={best_loss:.4f}")

    del model, optimizer, scaler
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"\nFine-tuning complete.")
print(f"Saved: {sorted([f for f in os.listdir(OUT_DIR) if 'v34' in f])}")
print(f"Best losses per fold: {['%.4f' % l for l in fold_results]}")

In [ ]:
# === CELL 8: UPLOAD AS birdclef-2026-perch-weights-v34 ===
import shutil, subprocess, json as _json

KAGGLE_USERNAME = os.environ.get('KAGGLE_USERNAME', 'chiragggg')
DATASET_SLUG    = 'birdclef-2026-perch-weights-v34'

_upload_dir = '/kaggle/working/upload_v34'
os.makedirs(_upload_dir, exist_ok=True)

_copied = []
for pt in Path(OUT_DIR).glob('perch_gru_v34_fold*.pt'):
    dst = os.path.join(_upload_dir, pt.name)
    shutil.copy2(str(pt), dst)
    _copied.append(pt.name)
print(f"Files to upload: {sorted(_copied)}")

if not _copied:
    print("ERROR: no v34 checkpoints found in", OUT_DIR)
else:
    _meta = {
        'title':    DATASET_SLUG,
        'id':       f'{KAGGLE_USERNAME}/{DATASET_SLUG}',
        'licenses': [{'name': 'CC0-1.0'}],
    }
    with open(os.path.join(_upload_dir, 'dataset-metadata.json'), 'w') as _mf:
        _json.dump(_meta, _mf, indent=2)

    _result = subprocess.run(
        ['kaggle', 'datasets', 'create', '-p', _upload_dir, '--dir-mode', 'zip'],
        capture_output=True, text=True,
    )
    print(_result.stdout)
    if _result.returncode != 0:
        print('STDERR:', _result.stderr)
        print('If dataset already exists, run:')
        print(f'  kaggle datasets version -p {_upload_dir} -m "v34 pseudo-label fine-tuning"')
    else:
        print(f'Upload complete: {KAGGLE_USERNAME}/{DATASET_SLUG}')
        print('Use in inference v30: attach as birdclef-2026-perch-weights-v34')
        print('Change checkpoint pattern to: perch_gru_v34_fold{i}.pt')